# Fix Neural Data from MATLAB Files

This notebook loads sorted spikes data from MATLAB files and creates a DataFrame with trial names and neural data.

In [1]:
import numpy as np
import pandas as pd
import re
from scipy.io import loadmat
from pathlib import Path
from tqdm import tqdm

monkey_name = 'yasmin' # 'fiona' or 'yasmin'

In [2]:
def parse_neural_data_from_mat(data):
    """
    Parse MATLAB dataStruct into a DataFrame with trial names and neural data.
    
    Parameters:
    -----------
    data : dict
        Dictionary loaded from MATLAB file using scipy.io.loadmat
    
    Returns:
    --------
    pd.DataFrame : DataFrame with columns ['trial_name', 'neural_data']
        neural_data is a dict with keys=neuron_id (int), values=spike_times (numpy array)
    """
    data_struct = data['dataStruct']
    num_trials = data_struct.shape[1]
    
    trial_names = []
    neural_data_list = []
    
    for i in range(num_trials):
        trial_entry = data_struct[0, i]
        
        # Extract trial name
        trial_name_obj = trial_entry[0]
        if hasattr(trial_name_obj, '__len__') and len(trial_name_obj) > 0:
            trial_name = str(trial_name_obj[0])
        else:
            trial_name = f"trial_{i}"
        
        # Extract spikes for all neurons
        spikes_obj = trial_entry[1]
        neural_data_dict = {}
        
        if hasattr(spikes_obj, 'shape') and len(spikes_obj.shape) > 1:
            num_neurons = spikes_obj.shape[1]
            
            for j in range(num_neurons):
                neuron_spikes = spikes_obj[0, j]
                
                # Flatten the spike times array
                if hasattr(neuron_spikes, 'flatten'):
                    spike_times = neuron_spikes.flatten()
                    # Only add non-empty spike trains
                    if len(spike_times) > 0:
                        neural_data_dict[j] = spike_times
                elif hasattr(neuron_spikes, '__len__') and len(neuron_spikes) > 0:
                    spike_times = np.array(neuron_spikes).flatten()
                    if len(spike_times) > 0:
                        neural_data_dict[j] = spike_times
        
        trial_names.append(trial_name)
        neural_data_list.append(neural_data_dict)
    
    # Create DataFrame
    df = pd.DataFrame({
        'trial_name': trial_names,
        'neural_data': neural_data_list
    })
    
    return df


In [3]:
# # Load the exemplar file
# data_dir = Path.cwd().parent / 'data' / 'fiona_sst' / 'sorted_spikes_in_session'
# mat_file = data_dir / 'fi211109_sorted_spikes.mat'

# print(f"Loading: {mat_file}")
# print(f"File exists: {mat_file.exists()}")

# if mat_file.exists():
#     data = loadmat(mat_file)
#     print(f"\nTop-level keys: {list(data.keys())}")
#     print(f"\ndataStruct shape: {data['dataStruct'].shape}")
#     print(f"dataStruct dtype: {data['dataStruct'].dtype}")
#     print(f"\nNumber of trials: {data['dataStruct'].shape[1]}")


# # Parse examplar data
# neural_df = parse_neural_data_from_mat(data)

# print(f"Created DataFrame with shape: {neural_df.shape}")
# print(f"\nFirst few rows:")
# print(neural_df.head())
# print(f"\nSample neural_data for first trial:")
# first_trial_data = neural_df.iloc[0]['neural_data']
# print(f"  Number of neurons with spikes: {len(first_trial_data)}")
# print(f"  Neuron IDs: {list(first_trial_data.keys())[:10]}...")  # Show first 10
# if len(first_trial_data) > 0:
#     first_neuron_id = list(first_trial_data.keys())[0]
#     print(f"  Example - Neuron {first_neuron_id} spike times: {first_trial_data[first_neuron_id]}")

In [4]:
# Aggregate neural data from all '*_sorted_spikes.mat' files under data/

data_root = Path.cwd().parent / 'data' / f'{monkey_name}_sst' / 'sorted_spikes_in_session'
pattern = f'{monkey_name[:2]}2*_sorted_spikes.mat'
all_files = sorted(list(data_root.rglob(pattern)))
print(f'Found {len(all_files)} files matching pattern "{pattern}" under {data_root}')

# Filter out hidden/system files (starting with ._) and validate session name format
session_pattern = re.compile(r'^(?:fi|ya)2\d{5}$')
valid_files = []
for f in all_files:
    session_name = f.stem.split('_sorted_spikes')[0]
    if ((f.name.startswith('._')) or (not session_pattern.match(session_name))):
        print(f'Skipping hidden/system file: {f.name}')
        continue
    valid_files.append(f)

print(f'\nProcessing {len(valid_files)} valid files of {len(all_files)} files found:\n')

dfs = []
for mat_file in tqdm(valid_files, desc='Processing files'):
    # print(f'Processing: {mat_file.relative_to(data_root)}')
    try:
        data_local = loadmat(mat_file)
    except Exception as e:
        print(f'  Failed to load {mat_file.name}: {e}')
        continue

    # Parse using the function defined earlier in this notebook
    df = parse_neural_data_from_mat(data_local)

    # Derive session name from filename (strip suffix '_sorted_spikes')
    session_name = mat_file.stem.split('_sorted_spikes')[0]
    df['session'] = session_name

    # Ensure trial_number exists as integer on each parsed df
    if 'trial_number' not in df.columns:
        df['trial_number'] = df['trial_name'].apply(lambda x: int(x.split('.')[-1]))
    else:
        # coerce to int if strings with leading zeros are present
        df['trial_number'] = df['trial_number'].astype(int)

    dfs.append(df)

# Concatenate all session DataFrames
if len(dfs) > 0:
    neural_df_all = pd.concat(dfs, ignore_index=True)
    print(f'Combined DataFrame shape: {neural_df_all.shape}')
    print(f'Total sessions: {neural_df_all["session"].unique().shape[0]}')
    print(f'Total trials: {len(neural_df_all)}')
else:
    print('No parsed files found; neural_df_all not created')

Found 57 files matching pattern "ya2*_sorted_spikes.mat" under /home/barak/Projects/population_analysis/data/yasmin_sst/sorted_spikes_in_session

Processing 57 valid files of 57 files found:



Processing files: 100%|██████████| 57/57 [00:22<00:00,  2.48it/s]

Combined DataFrame shape: (129254, 4)
Total sessions: 57
Total trials: 129254


In [5]:
neural_df_all

,trial_name,neural_data,session,trial_number
0,ya230501a.0001,{},ya230501,1
1,ya230501a.0002,{},ya230501,2
2,ya230501a.0003,{},ya230501,3
3,ya230501a.0004,{},ya230501,4
4,ya230501a.0005,{},ya230501,5
...,...,...,...,...
129249,ya230904a.2585,"{0: [29.6], 1: [89.1, 982.43, 1225.08, 1486.2,...",ya230904,2585
129250,ya230904a.2586,"{0: [1664.12, 2136.88], 1: [750.72, 2498.5, 25...",ya230904,2586
129251,ya230904a.2587,"{1: [333.07, 364.05, 888.77], 2: [122.05, 388....",ya230904,2587
129252,ya230904a.2588,"{1: [446.87], 2: [483.25, 627.3, 1253.35], 4: ...",ya230904,2588


In [6]:
base_path = Path.cwd().parent / 'data' 
file_path = base_path / 'csst_trials_pkls' / f'all_{monkey_name}_CSST_trials_df.pkl'

orig_df = pd.read_pickle(file_path)
orig_df

,blinks,dir,direction,filename,first_relevant_saccade,flags,go_cue,hPos,hVel,neural_data,...,ssd_number,stop_cue,trial_failed,trial_length,trial_name,trial_number,trial_session,type,vPos,vVel
0,None,0,R,ya230528a.0525,"[1523, 1598]",8206,1420,"[-12.35, -12.35, -12.35, -12.35, -12.35, -12.3...","[2.7566941723485194, 2.7566941723485194, 3.032...","{1: [90.65, 1290.95, 1364.25, 1872.33, 1935.37...",...,NaN,NaN,False,2571,GO_R,0525,ya230528a,GO,"[-0.45, -0.45, -0.45, -0.475, -0.475, -0.5, -0...","[2.0215757263889143, 2.0215757263889143, 2.113..."
1,None,180,L,ya230528a.0476,"[1680, 1753]",8206,1375,"[-0.15, -0.15, -0.175, -0.175, -0.175, -0.175,...","[-15.161817947916859, -15.161817947916859, -2....","{0: [1394.35, 2074.33], 1: [183.48, 232.45, 27...",...,NaN,NaN,False,2526,GO_L,0476,ya230528a,GO,"[1.1, 1.1, 1.075, 1.075, 1.075, 1.05, 1.075, 1...","[-28.3020601694448, -28.3020601694448, -13.048..."
2,None,0,R,ya230528a.1504,NaN,11278,1276,"[-0.225, -0.225, -0.225, -0.225, -0.225, -0.32...","[0.7351184459596053, 0.7351184459596053, 0.459...","{2: [69.68, 358.55, 504.93, 597.0, 661.85, 832...",...,1.0,1324.0,False,2024,STOP_R_SSD1,1504,ya230528a,STOP,"[-0.175, -0.175, -0.175, -0.175, -0.175, -0.17...","[-2.8485839780934703, -2.8485839780934703, -2...."
3,"[509, 568]",0,R,ya230528a.1499,"[1531, 1606]",8194,1344,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[5.7890577619318915, 5.7890577619318915, 3.308...","{2: [3.38, 436.4, 554.13, 910.75, 937.85, 1007...",...,NaN,NaN,True,2406,GO_R,1499,ya230528a,GO,"[-0.4, -0.4, -0.4, -0.4, -0.4, -0.45, -0.45, -...","[-0.5513388344697039, -0.5513388344697039, -0...."
4,None,180,L,ya230528a.1105,"[1775, 1851]",8206,1662,"[2.525, 2.525, 2.525, 2.4, 2.4, 2.3, 2.3, 2.3,...","[-39.32883685883888, -39.32883685883888, -39.3...","{1: [678.0, 845.65, 1164.3, 1219.3, 1269.05, 1...",...,NaN,NaN,False,2813,GO_L,1105,ya230528a,GO,"[-7.3, -7.3, -7.3, -6.25, -6.25, -5.625, -5.25...","[188.098432359914, 188.098432359914, 188.09843..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
123173,"[0, 46]",180,L,ya230719a.1943,"[1619, 1696]",8206,1351,"[0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.7...","[175.2338595556209, 175.2338595556209, 135.813...","{0: [1261.25, 1546.42, 1647.44, 1679.95, 1702....",...,NaN,NaN,False,2502,GO_L,1943,ya230719a,GO,"[-22.45, -22.45, -22.45, -22.45, -22.45, -22.4...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
123174,None,180,L,ya230719a.1747,"[1468, 1504]",8206,1212,"[8.075, 8.075, 8.025, 8.025, 8.025, 8.025, 8.0...","[-4.8701597044823846, -4.8701597044823846, -5....","{0: [605.47, 874.37, 922.04, 1050.139999999999...",...,1.0,1236.0,False,2363,CONT_L_SSD1,1747,ya230719a,CONT,"[9.95, 9.95, 9.95, 9.95, 9.95, 9.95, 9.975, 9....","[0.36755922297980265, 0.36755922297980265, 1.3..."
123175,None,0,R,ya230719a.0591,"[1587, 1662]",8206,1278,"[-9.2, -9.2, -9.2, -9.175, -9.175, -9.175, -9....","[-1.286457280429309, -1.286457280429309, -2.20...","{0: [1414.7, 1522.38, 1564.18, 1673.8000000000...",...,NaN,NaN,False,2429,GO_R,0591,ya230719a,GO,"[7.9, 7.9, 7.9, 7.925, 7.925, 7.925, 7.95, 7.9...","[-3.1242533953283225, -3.1242533953283225, -3...."
123176,None,180,L,ya230719a.1805,"[1577, 1665]",8206,1332,"[-8.075, -8.075, -8.075, -8.1, -8.1, -8.075, -...","[-2.113465532133865, -2.113465532133865, -2.20...","{0: [781.5, 831.15, 889.63, 915.68, 1024.0, 10...",...,1.0,1356.0,False,2483,CONT_L_SSD1,1805,ya230719a,CONT,"[-6.825, -6.825, -6.825, -6.8, -6.8, -6.825, -...","[-1.1945674746843584, -1.1945674746843584, -1...."


In [7]:
cell_ids = set([])
cell_ids_list = orig_df['neural_data'].apply(
    lambda x: list(x.keys()) if isinstance(x, dict) else x
)

for row in cell_ids_list:
    if isinstance(row, list):
        cell_ids.update(row)

lst = [i for i in cell_ids]
lst.sort()
lst == list(range(0, 50))

True

In [8]:
# Join neural_df_all with orig_df
# Select only trial_name and neural_data from neural_df_all
neural_subset = neural_df_all[['trial_name', 'neural_data']].copy()

# Rename neural_data to new_neural_data before merging
neural_subset.rename(columns={'neural_data': 'new_neural_data'}, inplace=True)

# Merge on trial_name (neural_df_all) = filename (orig_df)
merged_df = orig_df.merge(
    neural_subset,
    left_on='filename',
    right_on='trial_name',
    how='left'
)

print(f"Original df shape: {orig_df.shape}")
print(f"Neural df shape: {neural_df_all.shape}")
print(f"Merged df shape: {merged_df.shape}")
print(f"\nMerged df columns: {list(merged_df.columns)}")
print(f"\nRows with new_neural_data: {merged_df['new_neural_data'].notna().sum()}")
print(f"Rows without new_neural_data: {merged_df['new_neural_data'].isna().sum()}")

merged_df

Original df shape: (123178, 28)
Neural df shape: (129254, 4)
Merged df shape: (123178, 30)

Merged df columns: ['blinks', 'dir', 'direction', 'filename', 'first_relevant_saccade', 'flags', 'go_cue', 'hPos', 'hVel', 'neural_data', 'reaction_time', 'saccades', 'screen_rotation', 'segs_durations', 'segs_times', 'set', 'speed', 'ssd_len', 'ssd_number', 'stop_cue', 'trial_failed', 'trial_length', 'trial_name_x', 'trial_number', 'trial_session', 'type', 'vPos', 'vVel', 'trial_name_y', 'new_neural_data']

Rows with new_neural_data: 123178
Rows without new_neural_data: 0


,blinks,dir,direction,filename,first_relevant_saccade,flags,go_cue,hPos,hVel,neural_data,...,trial_failed,trial_length,trial_name_x,trial_number,trial_session,type,vPos,vVel,trial_name_y,new_neural_data
0,None,0,R,ya230528a.0525,"[1523, 1598]",8206,1420,"[-12.35, -12.35, -12.35, -12.35, -12.35, -12.3...","[2.7566941723485194, 2.7566941723485194, 3.032...","{1: [90.65, 1290.95, 1364.25, 1872.33, 1935.37...",...,False,2571,GO_R,0525,ya230528a,GO,"[-0.45, -0.45, -0.45, -0.475, -0.475, -0.5, -0...","[2.0215757263889143, 2.0215757263889143, 2.113...",ya230528a.0525,"{1: [90.65, 1290.95, 1364.25, 1872.33, 1935.38..."
1,None,180,L,ya230528a.0476,"[1680, 1753]",8206,1375,"[-0.15, -0.15, -0.175, -0.175, -0.175, -0.175,...","[-15.161817947916859, -15.161817947916859, -2....","{0: [1394.35, 2074.33], 1: [183.48, 232.45, 27...",...,False,2526,GO_L,0476,ya230528a,GO,"[1.1, 1.1, 1.075, 1.075, 1.075, 1.05, 1.075, 1...","[-28.3020601694448, -28.3020601694448, -13.048...",ya230528a.0476,"{0: [1394.35, 2074.33], 1: [183.48, 232.45, 27..."
2,None,0,R,ya230528a.1504,NaN,11278,1276,"[-0.225, -0.225, -0.225, -0.225, -0.225, -0.32...","[0.7351184459596053, 0.7351184459596053, 0.459...","{2: [69.68, 358.55, 504.93, 597.0, 661.85, 832...",...,False,2024,STOP_R_SSD1,1504,ya230528a,STOP,"[-0.175, -0.175, -0.175, -0.175, -0.175, -0.17...","[-2.8485839780934703, -2.8485839780934703, -2....",ya230528a.1504,"{2: [69.68, 358.55, 504.93, 597.0, 661.85, 832..."
3,"[509, 568]",0,R,ya230528a.1499,"[1531, 1606]",8194,1344,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[5.7890577619318915, 5.7890577619318915, 3.308...","{2: [3.38, 436.4, 554.13, 910.75, 937.85, 1007...",...,True,2406,GO_R,1499,ya230528a,GO,"[-0.4, -0.4, -0.4, -0.4, -0.4, -0.45, -0.45, -...","[-0.5513388344697039, -0.5513388344697039, -0....",ya230528a.1499,"{2: [3.38, 436.4, 554.13, 910.75, 937.85, 1007..."
4,None,180,L,ya230528a.1105,"[1775, 1851]",8206,1662,"[2.525, 2.525, 2.525, 2.4, 2.4, 2.3, 2.3, 2.3,...","[-39.32883685883888, -39.32883685883888, -39.3...","{1: [678.0, 845.65, 1164.3, 1219.3, 1269.05, 1...",...,False,2813,GO_L,1105,ya230528a,GO,"[-7.3, -7.3, -7.3, -6.25, -6.25, -5.625, -5.25...","[188.098432359914, 188.098432359914, 188.09843...",ya230528a.1105,"{1: [678.0, 845.65, 1164.3, 1219.3, 1269.05, 1..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
123173,"[0, 46]",180,L,ya230719a.1943,"[1619, 1696]",8206,1351,"[0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.7...","[175.2338595556209, 175.2338595556209, 135.813...","{0: [1261.25, 1546.42, 1647.44, 1679.95, 1702....",...,False,2502,GO_L,1943,ya230719a,GO,"[-22.45, -22.45, -22.45, -22.45, -22.45, -22.4...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",ya230719a.1943,"{0: [1261.25, 1546.42], 2: [1261.25], 3: [1095..."
123174,None,180,L,ya230719a.1747,"[1468, 1504]",8206,1212,"[8.075, 8.075, 8.025, 8.025, 8.025, 8.025, 8.0...","[-4.8701597044823846, -4.8701597044823846, -5....","{0: [605.47, 874.37, 922.04, 1050.139999999999...",...,False,2363,CONT_L_SSD1,1747,ya230719a,CONT,"[9.95, 9.95, 9.95, 9.95, 9.95, 9.95, 9.975, 9....","[0.36755922297980265, 0.36755922297980265, 1.3...",ya230719a.1747,"{0: [605.47], 2: [585.87, 796.65, 924.5, 1407...."
123175,None,0,R,ya230719a.0591,"[1587, 1662]",8206,1278,"[-9.2, -9.2, -9.2, -9.175, -9.175, -9.175, -9....","[-1.286457280429309, -1.286457280429309, -2.20...","{0: [1414.7, 1522.38, 1564.18, 1673.8000000000...",...,False,2429,GO_R,0591,ya230719a,GO,"[7.9, 7.9, 7.9, 7.925, 7.925, 7.925, 7.95, 7.9...","[-3.1242533953283225, -3.1242533953283225, -3....",ya230719a.0591,"{0: [1414.7, 1522.38, 1564.18, 1673.8, 1680.23..."
123176,None,180,L,ya230719a.1805,"[1577, 1665]",8206,1332,"[-8.075, -8.075, -8.075, -8.1, -8.1, -8.075, -...","[-2.113465532133865, -2.113465532133865, -2.20...","{0: [781.5, 831.15, 889.63, 915.68, 1024.0, 10...",...,False,2483,CONT_L_SSD1,1805,ya230719a,CONT,"[-6.825, -6.825, -6.825, -6.8, -6.8, -6.825, -...","[-1.1945674746843584, -1

In [9]:
# Drop old neural_data and trial_name_y columns, rename new_neural_data to neural_data
merged_df = merged_df.drop(columns=['neural_data', 'trial_name_y'])
merged_df = merged_df.rename(columns={'new_neural_data': 'neural_data'})

print(f"Updated merged_df shape: {merged_df.shape}")
print(f"Updated columns: {list(merged_df.columns)}")

merged_df

Updated merged_df shape: (123178, 28)
Updated columns: ['blinks', 'dir', 'direction', 'filename', 'first_relevant_saccade', 'flags', 'go_cue', 'hPos', 'hVel', 'reaction_time', 'saccades', 'screen_rotation', 'segs_durations', 'segs_times', 'set', 'speed', 'ssd_len', 'ssd_number', 'stop_cue', 'trial_failed', 'trial_length', 'trial_name_x', 'trial_number', 'trial_session', 'type', 'vPos', 'vVel', 'neural_data']


,blinks,dir,direction,filename,first_relevant_saccade,flags,go_cue,hPos,hVel,reaction_time,...,stop_cue,trial_failed,trial_length,trial_name_x,trial_number,trial_session,type,vPos,vVel,neural_data
0,None,0,R,ya230528a.0525,"[1523, 1598]",8206,1420,"[-12.35, -12.35, -12.35, -12.35, -12.35, -12.3...","[2.7566941723485194, 2.7566941723485194, 3.032...",103.0,...,NaN,False,2571,GO_R,0525,ya230528a,GO,"[-0.45, -0.45, -0.45, -0.475, -0.475, -0.5, -0...","[2.0215757263889143, 2.0215757263889143, 2.113...","{1: [90.65, 1290.95, 1364.25, 1872.33, 1935.38..."
1,None,180,L,ya230528a.0476,"[1680, 1753]",8206,1375,"[-0.15, -0.15, -0.175, -0.175, -0.175, -0.175,...","[-15.161817947916859, -15.161817947916859, -2....",305.0,...,NaN,False,2526,GO_L,0476,ya230528a,GO,"[1.1, 1.1, 1.075, 1.075, 1.075, 1.05, 1.075, 1...","[-28.3020601694448, -28.3020601694448, -13.048...","{0: [1394.35, 2074.33], 1: [183.48, 232.45, 27..."
2,None,0,R,ya230528a.1504,NaN,11278,1276,"[-0.225, -0.225, -0.225, -0.225, -0.225, -0.32...","[0.7351184459596053, 0.7351184459596053, 0.459...",NaN,...,1324.0,False,2024,STOP_R_SSD1,1504,ya230528a,STOP,"[-0.175, -0.175, -0.175, -0.175, -0.175, -0.17...","[-2.8485839780934703, -2.8485839780934703, -2....","{2: [69.68, 358.55, 504.93, 597.0, 661.85, 832..."
3,"[509, 568]",0,R,ya230528a.1499,"[1531, 1606]",8194,1344,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[5.7890577619318915, 5.7890577619318915, 3.308...",187.0,...,NaN,True,2406,GO_R,1499,ya230528a,GO,"[-0.4, -0.4, -0.4, -0.4, -0.4, -0.45, -0.45, -...","[-0.5513388344697039, -0.5513388344697039, -0....","{2: [3.38, 436.4, 554.13, 910.75, 937.85, 1007..."
4,None,180,L,ya230528a.1105,"[1775, 1851]",8206,1662,"[2.525, 2.525, 2.525, 2.4, 2.4, 2.3, 2.3, 2.3,...","[-39.32883685883888, -39.32883685883888, -39.3...",113.0,...,NaN,False,2813,GO_L,1105,ya230528a,GO,"[-7.3, -7.3, -7.3, -6.25, -6.25, -5.625, -5.25...","[188.098432359914, 188.098432359914, 188.09843...","{1: [678.0, 845.65, 1164.3, 1219.3, 1269.05, 1..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
123173,"[0, 46]",180,L,ya230719a.1943,"[1619, 1696]",8206,1351,"[0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.7...","[175.2338595556209, 175.2338595556209, 135.813...",268.0,...,NaN,False,2502,GO_L,1943,ya230719a,GO,"[-22.45, -22.45, -22.45, -22.45, -22.45, -22.4...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","{0: [1261.25, 1546.42], 2: [1261.25], 3: [1095..."
123174,None,180,L,ya230719a.1747,"[1468, 1504]",8206,1212,"[8.075, 8.075, 8.025, 8.025, 8.025, 8.025, 8.0...","[-4.8701597044823846, -4.8701597044823846, -5....",256.0,...,1236.0,False,2363,CONT_L_SSD1,1747,ya230719a,CONT,"[9.95, 9.95, 9.95, 9.95, 9.95, 9.95, 9.975, 9....","[0.36755922297980265, 0.36755922297980265, 1.3...","{0: [605.47], 2: [585.87, 796.65, 924.5, 1407...."
123175,None,0,R,ya230719a.0591,"[1587, 1662]",8206,1278,"[-9.2, -9.2, -9.2, -9.175, -9.175, -9.175, -9....","[-1.286457280429309, -1.286457280429309, -2.20...",309.0,...,NaN,False,2429,GO_R,0591,ya230719a,GO,"[7.9, 7.9, 7.9, 7.925, 7.925, 7.925, 7.95, 7.9...","[-3.1242533953283225, -3.1242533953283225, -3....","{0: [1414.7, 1522.38, 1564.18, 1673.8, 1680.23..."
123176,None,180,L,ya230719a.1805,"[1577, 1665]",8206,1332,"[-8.075, -8.075, -8.075, -8.1, -8.1, -8.075, -...","[-2.113465532133865, -2.113465532133865, -2.20...",245.0,...,1356.0,False,2483,CONT_L_SSD1,1805,ya230719a,CONT,"[-6.825, -6.825, -6.825, -6.8, -6.8, -6.825, -...","[-1.1945674746843584, -1.1945674746843584, -1....","{0: [781.5], 4: [564.0, 625.4, 943.28, 1384.7,..."


In [10]:
# Rename trial_name_x to trial_name
merged_df = merged_df.rename(columns={'trial_name_x': 'trial_name'})

# Reorder columns to match orig_df
orig_columns = list(orig_df.columns)
merged_df = merged_df[orig_columns]

print(f"Updated merged_df shape: {merged_df.shape}")
print(f"Original df columns: {list(orig_df.columns)}")
print(f"Merged df columns: {list(merged_df.columns)}")
print(f"\nColumns match: {list(orig_df.columns) == list(merged_df.columns)}")

merged_df

Updated merged_df shape: (123178, 28)
Original df columns: ['blinks', 'dir', 'direction', 'filename', 'first_relevant_saccade', 'flags', 'go_cue', 'hPos', 'hVel', 'neural_data', 'reaction_time', 'saccades', 'screen_rotation', 'segs_durations', 'segs_times', 'set', 'speed', 'ssd_len', 'ssd_number', 'stop_cue', 'trial_failed', 'trial_length', 'trial_name', 'trial_number', 'trial_session', 'type', 'vPos', 'vVel']
Merged df columns: ['blinks', 'dir', 'direction', 'filename', 'first_relevant_saccade', 'flags', 'go_cue', 'hPos', 'hVel', 'neural_data', 'reaction_time', 'saccades', 'screen_rotation', 'segs_durations', 'segs_times', 'set', 'speed', 'ssd_len', 'ssd_number', 'stop_cue', 'trial_failed', 'trial_length', 'trial_name', 'trial_number', 'trial_session', 'type', 'vPos', 'vVel']

Columns match: True


,blinks,dir,direction,filename,first_relevant_saccade,flags,go_cue,hPos,hVel,neural_data,...,ssd_number,stop_cue,trial_failed,trial_length,trial_name,trial_number,trial_session,type,vPos,vVel
0,None,0,R,ya230528a.0525,"[1523, 1598]",8206,1420,"[-12.35, -12.35, -12.35, -12.35, -12.35, -12.3...","[2.7566941723485194, 2.7566941723485194, 3.032...","{1: [90.65, 1290.95, 1364.25, 1872.33, 1935.38...",...,NaN,NaN,False,2571,GO_R,0525,ya230528a,GO,"[-0.45, -0.45, -0.45, -0.475, -0.475, -0.5, -0...","[2.0215757263889143, 2.0215757263889143, 2.113..."
1,None,180,L,ya230528a.0476,"[1680, 1753]",8206,1375,"[-0.15, -0.15, -0.175, -0.175, -0.175, -0.175,...","[-15.161817947916859, -15.161817947916859, -2....","{0: [1394.35, 2074.33], 1: [183.48, 232.45, 27...",...,NaN,NaN,False,2526,GO_L,0476,ya230528a,GO,"[1.1, 1.1, 1.075, 1.075, 1.075, 1.05, 1.075, 1...","[-28.3020601694448, -28.3020601694448, -13.048..."
2,None,0,R,ya230528a.1504,NaN,11278,1276,"[-0.225, -0.225, -0.225, -0.225, -0.225, -0.32...","[0.7351184459596053, 0.7351184459596053, 0.459...","{2: [69.68, 358.55, 504.93, 597.0, 661.85, 832...",...,1.0,1324.0,False,2024,STOP_R_SSD1,1504,ya230528a,STOP,"[-0.175, -0.175, -0.175, -0.175, -0.175, -0.17...","[-2.8485839780934703, -2.8485839780934703, -2...."
3,"[509, 568]",0,R,ya230528a.1499,"[1531, 1606]",8194,1344,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[5.7890577619318915, 5.7890577619318915, 3.308...","{2: [3.38, 436.4, 554.13, 910.75, 937.85, 1007...",...,NaN,NaN,True,2406,GO_R,1499,ya230528a,GO,"[-0.4, -0.4, -0.4, -0.4, -0.4, -0.45, -0.45, -...","[-0.5513388344697039, -0.5513388344697039, -0...."
4,None,180,L,ya230528a.1105,"[1775, 1851]",8206,1662,"[2.525, 2.525, 2.525, 2.4, 2.4, 2.3, 2.3, 2.3,...","[-39.32883685883888, -39.32883685883888, -39.3...","{1: [678.0, 845.65, 1164.3, 1219.3, 1269.05, 1...",...,NaN,NaN,False,2813,GO_L,1105,ya230528a,GO,"[-7.3, -7.3, -7.3, -6.25, -6.25, -5.625, -5.25...","[188.098432359914, 188.098432359914, 188.09843..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
123173,"[0, 46]",180,L,ya230719a.1943,"[1619, 1696]",8206,1351,"[0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.7...","[175.2338595556209, 175.2338595556209, 135.813...","{0: [1261.25, 1546.42], 2: [1261.25], 3: [1095...",...,NaN,NaN,False,2502,GO_L,1943,ya230719a,GO,"[-22.45, -22.45, -22.45, -22.45, -22.45, -22.4...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
123174,None,180,L,ya230719a.1747,"[1468, 1504]",8206,1212,"[8.075, 8.075, 8.025, 8.025, 8.025, 8.025, 8.0...","[-4.8701597044823846, -4.8701597044823846, -5....","{0: [605.47], 2: [585.87, 796.65, 924.5, 1407....",...,1.0,1236.0,False,2363,CONT_L_SSD1,1747,ya230719a,CONT,"[9.95, 9.95, 9.95, 9.95, 9.95, 9.95, 9.975, 9....","[0.36755922297980265, 0.36755922297980265, 1.3..."
123175,None,0,R,ya230719a.0591,"[1587, 1662]",8206,1278,"[-9.2, -9.2, -9.2, -9.175, -9.175, -9.175, -9....","[-1.286457280429309, -1.286457280429309, -2.20...","{0: [1414.7, 1522.38, 1564.18, 1673.8, 1680.23...",...,NaN,NaN,False,2429,GO_R,0591,ya230719a,GO,"[7.9, 7.9, 7.9, 7.925, 7.925, 7.925, 7.95, 7.9...","[-3.1242533953283225, -3.1242533953283225, -3...."
123176,None,180,L,ya230719a.1805,"[1577, 1665]",8206,1332,"[-8.075, -8.075, -8.075, -8.1, -8.1, -8.075, -...","[-2.113465532133865, -2.113465532133865, -2.20...","{0: [781.5], 4: [564.0, 625.4, 943.28, 1384.7,...",...,1.0,1356.0,False,2483,CONT_L_SSD1,1805,ya230719a,CONT,"[-6.825, -6.825, -6.825, -6.8, -6.8, -6.825, -...","[-1.1945674746843584, -1.1945674746843584, -1...."


In [11]:
merged_df.to_pickle(file_path)

In [ ]:
# monkey = 'fiona'
# save_path = Path.cwd().parent / 'data' / 'unified_cell_trial_data'
# pickle_file = save_path / f'unified_{monkey}_cell_trial_data.pkl'

# cells_db = pd.read_pickle(pickle_file)
# cells_db['maestro_ID'].value_counts().sort_index()